In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
df_m = session.sql("SELECT * FROM analytics_restaurant_monthly").to_pandas()
df_c = session.sql("SELECT * FROM analytics_restaurant_checkins_monthly").to_pandas()
df_e = session.sql("SELECT * FROM analytics_restaurant_engagement").to_pandas()
df_r = session.sql("SELECT * FROM analytics_restaurant_rating_trend").to_pandas()

In [ ]:
df_m["YEAR_MONTH"] = pd.to_datetime(df_m["YEAR_MONTH"] + "-01")
df_c["YEAR_MONTH"] = pd.to_datetime(df_c["YEAR_MONTH"] + "-01")
df_r["YEAR_MONTH"] = pd.to_datetime(df_r["YEAR_MONTH"] + "-01")
df_e["YEAR_MONTH"] = pd.to_datetime(df_e["YEAR_MONTH"] + "-01")

In [ ]:
print(df_r.info())
print(df_e.info())
print(df_c.info())
print(df_m.info())

In [ ]:
print(df_r.describe())
print(df_e.describe())
print(df_c.describe())
print(df_m.describe())

In [ ]:
print(df_e.isna().sum())
print(df_r.isna().sum())
print(df_c.isna().sum())
print(df_m.isna().sum())

In [ ]:
df_m["RATING_VOLATILITY"] = df_m["RATING_VOLATILITY"].fillna(0)

In [ ]:
df_m[df_m["RATING_VOLATILITY"].isna()]

In [ ]:
df_m["RATING_VOLATILITY"].value_counts()

In [ ]:
df_r.info()

In [ ]:
df = df_r.merge(df_e, on=["BUSINESS_ID","YEAR_MONTH"], how="inner") \
        .merge(df_c, on=["BUSINESS_ID","YEAR_MONTH"], how="inner")


In [ ]:
df["ENGAGEMENT_CHANGE"] = df.groupby("BUSINESS_ID")["ENGAGEMENT_SCORE"].diff()


In [ ]:
df["RATING_DROP_FLAG"] = df["RATING_CHANGE_3M"].apply(lambda x: 1 if x < -0.5 else 0)

In [ ]:
df["ENGAGEMENT_DROP_FLAG"] = df["ENGAGEMENT_CHANGE"].apply(lambda x: 1 if x < 0 else 0)

In [ ]:
df.head()

In [ ]:
df.drop("CHECKINS_COUNT_y", axis=1, inplace=True)

In [ ]:
df.rename(columns={"CHECKINS_COUNT_x": "CHECKINS_COUNT"}, inplace=True)


In [ ]:
df.drop("CHECKINS_COUNT_y", axis=1, inplace=True)

In [ ]:
corr = df[["REVIEWS_COUNT","CHECKINS_COUNT","ENGAGEMENT_SCORE","RATING_CHANGE_3M","RATING_DROP_FLAG","ENGAGEMENT_DROP_FLAG"]].corr()
corr

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df_m.groupby("YEAR_MONTH")["REVIEWS_COUNT"].sum().plot()
plt.title("Reviews Volume Trend")
plt.xlabel("Month")
plt.ylabel("Reviews")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df_c.groupby("YEAR_MONTH")["CHECKINS_COUNT"].sum().plot()
plt.title("Check-ins Volume Trend")
plt.xlabel("Month")
plt.ylabel("Check-ins")
plt.show()

In [ ]:
df_m["RATING_VOLATILITY"].dropna().plot(kind="hist", bins=40)
plt.title("Rating Volatility Distribution")
plt.xlabel("Volatility")
plt.ylabel("Restaurants")
plt.show()

In [ ]:
sample = df[df["REVIEWS_COUNT"]>20]
plt.scatter(sample["ENGAGEMENT_SCORE"], sample["RATING_CHANGE_3M"], alpha=0.3)
plt.title("Engagement vs Rating Change (3M)")
plt.xlabel("Engagement")
plt.ylabel("Rating Change")
plt.show()

In [ ]:
risk = df[
    (df["RATING_DROP_FLAG"]==1) &
    (df["ENGAGEMENT_DROP_FLAG"]==1)
]

risk[["BUSINESS_ID","YEAR_MONTH","RATING_CHANGE_3M","ENGAGEMENT_SCORE","ENGAGEMENT_CHANGE"]].head(10)


In [ ]:
top10 = df_m.groupby("BUSINESS_ID")["REVIEWS_COUNT"].sum().sort_values(ascending=False).head(10)
top10.plot(kind="bar")
plt.title("Top 10 Restaurants by Review Volume")
plt.xlabel("Business ID")
plt.ylabel("Total Reviews")
plt.show()

In [ ]:
top10_c = df_c.groupby("BUSINESS_ID")["CHECKINS_COUNT"].sum().sort_values(ascending=False).head(10)
top10_c.plot(kind="bar")
plt.title("Top 10 Restaurants by Check-ins")
plt.xlabel("Business ID")
plt.ylabel("Total Check-ins")
plt.show()

In [ ]:
df_r["RATING_CHANGE_3M"].plot(kind="hist")
plt.title("3-Month Rating Change Distribution")
plt.xlabel("Rating Change")
plt.ylabel("Restaurants")
plt.show()

In [ ]:
df_e["ENGAGEMENT_SCORE"].plot(kind="hist")
plt.title("Restaurant Engagement Score Distribution")
plt.xlabel("Engagement Score")
plt.ylabel("Restaurants")
plt.show()

In [ ]:
sample = df_m[df_m["REVIEWS_COUNT"]>20]
plt.scatter(sample["REVIEWS_COUNT"], sample["RATING_VOLATILITY"])
plt.title("Review Count vs Rating Volatility")
plt.xlabel("Reviews")
plt.ylabel("Volatility")
plt.show()

In [ ]:
df.head()

In [ ]:
df["RATING_CHANGE_3M"] = df["RATING_CHANGE_3M"].fillna(0)
df["ENGAGEMENT_CHANGE"] = df["ENGAGEMENT_CHANGE"].fillna(0)
df["RATING_CHG_Z"] = (df["RATING_CHANGE_3M"] - df["RATING_CHANGE_3M"].mean()) / df["RATING_CHANGE_3M"].std()
df["ENGAGEMENT_Z"] = (df["ENGAGEMENT_CHANGE"] - df["ENGAGEMENT_CHANGE"].mean()) / df["ENGAGEMENT_CHANGE"].std()


In [ ]:
df = df.merge(df_m[["BUSINESS_ID", "YEAR_MONTH", "RATING_VOLATILITY"]], 
              on=["BUSINESS_ID", "YEAR_MONTH"], 
              how="left")

In [ ]:
df["RATING_VOLATILITY"] = df["RATING_VOLATILITY"].fillna(0)

In [ ]:
df["RATING_VOL_Z"] = (df["RATING_VOLATILITY"] - df["RATING_VOLATILITY"].mean()) / df["RATING_VOLATILITY"].std()


In [ ]:
def calc_risk(row):
    score = 0
    if row["RATING_CHG_Z"] < -1: score += 2
    if row["ENGAGEMENT_Z"] < -1: score += 2
    if row["RATING_VOL_Z"] > 1: score += 1
    return score

df["RISK_SCORE"] = df.apply(calc_risk, axis=1)

In [ ]:
df[["BUSINESS_ID","YEAR_MONTH","RISK_SCORE"]].head(10)

In [ ]:
df["RISK_SCORE"].plot(kind="hist", bins=30)
plt.title("Risk Score Distribution")
plt.xlabel("Risk Score")
plt.ylabel("Restaurants")
plt.show()

In [ ]:
df.groupby("BUSINESS_ID")["RISK_SCORE"].max().sort_values(ascending=False).head(10)

In [ ]:
df.groupby("BUSINESS_ID")["RISK_SCORE"].max().sort_values(ascending=False).value_counts()

UPDATE analytics_restaurant_monthly
SET YEAR_MONTH = TO_CHAR(
    TO_DATE(SUBSTR(YEAR_MONTH, 1, 7) || '-01'),
    'YYYY-MM'
);

UPDATE analytics_restaurant_checkins_monthly
SET YEAR_MONTH = TO_CHAR(
    DATE_TRUNC('month', TO_TIMESTAMP_NTZ(YEAR_MONTH || '-01 00:00:00')),
    'YYYY-MM'
);


UPDATE analytics_restaurant_rating_trend
SET YEAR_MONTH = TO_CHAR(
    TO_DATE(YEAR_MONTH || '-01'),
    'YYYY-MM'
);

In [ ]:
df.head(4)

In [ ]:
df["YEAR_MONTH"] = df["YEAR_MONTH"].dt.to_period("M").astype(str)

In [ ]:
df.columns

In [ ]:
session.write_pandas(df, "FINAL_ANALYTICS_RISK_SCORES", auto_create_table=True, overwrite=True)


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS_RESTAURANT_RISK_FINAL AS
SELECT BUSINESS_ID, YEAR_MONTH, RISK_SCORE
FROM FINAL_ANALYTICS_RISK_SCORES
ORDER BY BUSINESS_ID, YEAR_MONTH;

In [ ]:
df_b = session.sql("SELECT * FROM stg_yelp_business").to_pandas()


In [ ]:
df_city_risk = df.merge(df_b, on="BUSINESS_ID", how="left")
df_city_risk.groupby("CITY")["RISK_SCORE"].mean().sort_values(ascending=False).head(10)

In [ ]:
df_cat = df_city_risk.copy()
df_cat = df_cat.explode("CATEGORIES")
df_cat.groupby("CATEGORIES")["RISK_SCORE"].mean().sort_values(ascending=False).head(10)

In [ ]:
df_city_risk.columns

In [ ]:
closed = df_city_risk[df_city_risk["OPENED"] == 0]
closed[["ENGAGEMENT_SCORE","RISK_SCORE"]].corr()

In [ ]:
crash = df[
    (df["ENGAGEMENT_CHANGE"] < df["ENGAGEMENT_CHANGE"].quantile(0.01))  # bottom 1%
]
crash[["BUSINESS_ID","YEAR_MONTH","ENGAGEMENT_CHANGE","RISK_SCORE"]].head(10)


In [ ]:
instable = df[
    (df["RATING_VOLATILITY"] > df["RATING_VOLATILITY"].quantile(0.95))  # top 5% volatility
]
instable[["BUSINESS_ID","YEAR_MONTH","RATING_VOLATILITY","RISK_SCORE"]].head(10)


In [ ]:
one = df[df["BUSINESS_ID"] == df["BUSINESS_ID"].iloc[0]]
one[["YEAR_MONTH","REVIEWS_COUNT","CHECKINS_COUNT","ENGAGEMENT_SCORE","RISK_SCORE"]]

In [ ]:
df_closed = df_city_risk[df_city_risk["OPENED"] == 0]
df_closed.groupby("YEAR_MONTH")["ENGAGEMENT_CHANGE"].mean().plot()
plt.title("Avg Engagement Change Over Time (Closed Restaurants)")
plt.xlabel("Month")
plt.ylabel("Engagement Change")
plt.show()